# AgentOps Lab 04 - Tool engineering

This notebook turns a vague, dangerous operations tool into narrow, validated, auditable tools. It demonstrates why agent reliability often depends less on clever prompting and more on tool boundaries.


## The dangerous tool

```python
admin_api(command: str)
```

This single tool can query logs, restart services, delete records, deploy software, send notifications, and change config. It is dangerous because the schema hides intent. A model can put anything in the command string, authorization cannot easily distinguish read-only from destructive actions, and failures are hard to classify.

```mermaid
flowchart TD
    A["admin_api(command)"] --> B["query logs"]
    A --> C["restart service"]
    A --> D["delete records"]
    A --> E["deploy software"]
    A --> F["send notifications"]
    A --> G["change config"]
```


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.tool_engineering import (
    PermissionDenied,
    RestartRequest,
    admin_api,
    compare_bad_and_good_tools,
    create_incident_ticket,
    query_logs,
    restart_service,
    retry_policy,
    run_with_retry,
)


In [ ]:
admin_api("restart checkout and delete failed payment records")


## Refactor into narrow tools

A better design separates read-only tools from write tools and gives each tool a typed contract:

- `query_logs(service, time_range_minutes, severity)`
- `get_recent_deployments(service)`
- `restart_service(service, reason, incident_id)`
- `create_incident_ticket(title, severity, evidence)`

```mermaid
flowchart LR
    A["Agent request"] --> B{"Read-only?"}
    B -- "Yes" --> C["query_logs / get_recent_deployments"]
    B -- "No" --> D["Validate structured request"]
    D --> E{"Human approval?"}
    E -- "Approved" --> F["restart_service"]
    E -- "Denied or missing" --> G["escalate"]
```


## Structured validation

In a real SDK integration, you would commonly use Pydantic-style validation:

```python
from typing import Literal
from pydantic import BaseModel, Field

class RestartRequest(BaseModel):
    service: Literal["checkout", "payments", "catalog"]
    reason: str = Field(min_length=20)
    incident_id: str
```

This repository keeps the executable lab dependency-free, so the Python module uses an equivalent dataclass validator. The principle is the same: invalid services, short reasons, and malformed incident IDs should fail before a tool touches infrastructure.


In [ ]:
query_logs("checkout", time_range_minutes=60, severity="ERROR")


In [ ]:
create_incident_ticket(
    title="European checkout 3DS failures",
    severity="sev2",
    evidence=["eu-west 3DS callback errors", "active checkout payment incident"],
)


## Simulate tool failures

A reliable tool contract includes predictable failure classes. In this lab:

- `ToolTimeout` and `RateLimit` are retryable.
- `PermissionDenied` escalates to a human.
- `InvalidService` and validation errors stop the run.


In [ ]:
run_with_retry(max_attempts=3, attempts_before_success=2)


In [ ]:
try:
    restart_service(RestartRequest("checkout", "Need approval for regional checkout recovery", "INC-1042"))
except Exception as exc:
    print(type(exc).__name__, "->", retry_policy(exc))


## Exercise

- Add an `update_feature_flag(service, flag_name, desired_state, incident_id)` tool. What validation and approval does it need?
- Add a `RateLimit` simulation and use exponential backoff in the retry policy.
- Write three bad `admin_api` commands and refactor each into a safe narrow tool.
- Decide which tools should be available to a support assistant versus an on-call engineering assistant.

References: [OpenAI Agents SDK tools](https://openai.github.io/openai-agents-python/tools/), [OpenAI Agents SDK guardrails](https://openai.github.io/openai-agents-python/guardrails/), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).
